> **Notebook-first lesson.** Run cells in order. The final activity is designed to be changed and rerun.

## Mathematical Framework

Math companions for this lesson:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Do not stop at the API surface. Identify the **spaces/vectors involved, objective or probability model, local derivatives, matrix shapes, and approximation assumptions**.

# Lesson 30: Debugging PyTorch models

Most deep-learning bugs are not solved by "train longer."

## Debugging ladder
1. Verify input shapes.
2. Verify target shapes/dtypes.
3. Inspect min/max/mean/std.
4. Run one forward pass.
5. Inspect loss before training.
6. Overfit a tiny batch.
7. Inspect gradient magnitudes.
8. Check for NaN/Inf.
9. Verify train/eval mode.
10. Confirm data split and labels.

## Useful checks


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED)
X_train = torch.randn(160, 2)
y_train = (X_train[:, 0] + .5 * X_train[:, 1] > 0).long()
X_val = torch.randn(60, 2)
y_val = (X_val[:, 0] + .5 * X_val[:, 1] > 0).long()
device = torch.device('cpu')
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.01)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
loss_fn(model(X_train), y_train).backward()


In [ ]:
for name, p in model.named_parameters():
    if p.grad is not None:
        print(name, p.grad.norm().item())



## Common mistakes
- wrong output dimension
- applying softmax before CrossEntropyLoss
- forgetting optimizer.zero_grad()
- target dtype is float instead of long for class indices
- model on GPU but data on CPU
- evaluation with dropout still active
- data leakage
- normalization computed using test data

## Exercise
Create three deliberately broken training scripts, one each for shape, gradient and data problems. Diagnose them systematically.


## Runnable activity
Run this experiment. Then change one architectural, data, or optimization choice and compare.

In [ ]:
import torch
from torch import nn
torch.manual_seed(0)
model=nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,3))
X=torch.randn(6,4); y=torch.randint(0,3,(6,))
loss=nn.CrossEntropyLoss()(model(X),y)
loss.backward()
print("loss",float(loss))
for name,p in model.named_parameters():
    print(name,"shape",tuple(p.shape),"grad norm",float(p.grad.norm()))
print("finite loss:",torch.isfinite(loss).item())

## Explanation checkpoint
Add a Markdown cell that explains the tensor shapes, the mechanism being tested, and what changed when you modified the experiment.